# WANDS query understanding on a free Kaggle T4

Runs the LLM step of `wands-search` (spelling, attribute extraction, rewriting) with
**Qwen2.5-7B-Instruct in 4-bit** on a free Kaggle GPU, because the laptop has no GPU.

Settings: *Accelerator: GPU T4 x1*, *Internet: on*. Then **Run All** (about 15–20 minutes).
Download `/kaggle/working/qwen2.5-7b-instruct-4bit.jsonl` and put it in `results/llm/` in the repo.
`scripts/run_llm.py --backend cache` then reads it, exactly like a Groq or Ollama cache file.

The prompt below is a copy of `PROMPT` in `wands_search/llm.py`. Keep the two the same.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import json, time, urllib.request, io, csv, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"
OUT = "/kaggle/working/qwen2.5-7b-instruct-4bit.jsonl"
PROMPT = """You help a furniture and home-goods store understand search queries.
For the query below, return ONLY a JSON object with these keys:
  "corrected": the query with spelling mistakes fixed; identical to the input if there are none.
               Do not change brand or product-line names you do not recognise.
  "product_type": the kind of product being searched for, as a short noun phrase, or null
  "color": colour words in the query, or null
  "material": material words in the query, or null
  "size": size or dimension words in the query (e.g. "queen", "24 inch", "small"), or null
  "style": style words in the query (e.g. "modern", "farmhouse"), or null
  "rewrite": the query rewritten as a store's catalogue would describe the product. Keep every
             constraint in the query, do not add new ones, at most 10 words.
Only use words that are in the query for color, material, size and style; do not guess.

Query: {query}"""

raw = urllib.request.urlopen("https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/query.csv").read().decode("utf-8")
queries = [(int(r["query_id"]), r["query"]) for r in csv.DictReader(io.StringIO(raw), delimiter="	")]
print(len(queries), queries[:3])

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4"))
model.eval()

In [ ]:
def prompts(batch):
    return [tok.apply_chat_template([{"role": "user", "content": PROMPT.format(query=q)}],
                                    tokenize=False, add_generation_prompt=True) for _, q in batch]

BATCH = 16
with open(OUT, "w", encoding="utf-8") as fh:
    for i in range(0, len(queries), BATCH):
        batch = queries[i:i + BATCH]
        enc = tok(prompts(batch), return_tensors="pt", padding=True).to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=200, do_sample=False)
        sec = (time.perf_counter() - t0) / len(batch)   # batched: per-query share of the batch time
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for (qid, q), t in zip(batch, texts):
            fh.write(json.dumps({"query_id": qid, "query": q, "raw": t, "seconds": round(sec, 3)}) + "
")
        print(i + len(batch), f"{sec:.2f}s/query", texts[0][:120].replace("
", " "))

In [ ]:
lines = open(OUT, encoding="utf-8").read().splitlines()
print(len(lines)); print(lines[0])